<a href="https://colab.research.google.com/github/annagiacometti/cultural-data-framework-photography-festivals/blob/main/proof_of_concept_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline sperimentale per l'estrazione automatica di dati culturali

## *Proof of Concept su festival fotografici italiani*

Notebook sviluppato nell'ambito della tesi di Laurea Magistrale:

"Dalla frammentazione alla strutturazione del dato culturale: sviluppo di un framework metodologico e sperimentazione di strumenti basati su intelligenza artificiale. Il caso dei festival fotografici italiani"

---

Anna Giacometti

Università degli Studi di Palermo -
Corso di Laurea Magistrale in Digital Humanities per l'Industria Culturale

## Introduzione

Questo notebook implementa la pipeline sperimentale descritta nel Capitolo 5 della tesi.

L'obiettivo non consiste nella valutazione delle prestazioni di uno specifico modello linguistico, ma nella verifica della fattibilità di una pipeline basata su intelligenza artificiale per supportare la raccolta e la strutturazione di dati culturali provenienti da fonti web non strutturate.

La pipeline è progettata a partire dal framework metodologico sviluppato nella ricerca (modello concettuale, modello dati e codebook) ed è applicata a un campione controllato di festival fotografici italiani.

# 1. Importazione delle librerie e configurazione dell'ambiente

In [46]:
# ==========================================================
# LIBRERIE
# ==========================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)
from collections import Counter

!git clone https://github.com/annagiacometti/cultural-data-framework-photography-festivals.git
# Definizione dei percorsi delle risorse

BASE_PATH = "cultural-data-framework-photography-festivals/pipeline_ia"

ISTAT_PATH = f"{BASE_PATH}/data/vocabolario_istat_comuni_italiani.xlsx"

# ==========================================================
# CONFIGURAZIONE MODELLO
# ==========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id
).to(device)

print("Device:", device)



Cloning into 'cultural-data-framework-photography-festivals'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 205 (delta 3), reused 0 (delta 0), pack-reused 187 (from 1)
Receiving objects: 100% (205/205), 2.05 MiB | 4.65 MiB/s, done.
Resolving deltas: 100% (53/53), done.


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Device: cuda


In [48]:
!find cultural-data-framework-photography-festivals -name "*.xlsx"

cultural-data-framework-photography-festivals/dataset/dataset_manual_snapshot_2026.xlsx
cultural-data-framework-photography-festivals/pipeline_ia/data/vocabolario_istat_comuni_italiani.xlsx


# 2. Acquisizione e preparazione delle fonti digitali

##2.1 Funzione di scraping e pulizia HTML

La funzione acquisisce il contenuto HTML delle pagine web selezionate e applica una prima fase di pulizia, eliminando elementi non informativi della struttura della pagina (script, stili, menu di navigazione, intestazioni e footer). Il risultato è un testo normalizzato utilizzabile nelle successive fasi della pipeline.

In [26]:
def scrape_page(url):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup(
            [
                "script",
                "style",
                "nav",
                "footer",
                "header"
            ]
        ):
            tag.extract()

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        return text

    except Exception as e:

        print(e)

        return ""

##2.2 Preparazione del testo e suddivisione in chunk

Il testo estratto viene successivamente convertito in sequenze di token e suddiviso mediante una strategia di sliding window con sovrapposizione, al fine di adattare documenti estesi ai limiti di input del modello linguistico.

In [27]:
def prepare_pipeline(urls):

    pages = {}

    print("=" * 70)
    print("HARVESTING DELLE FONTI")
    print("=" * 70)

    for page_name, url in urls.items():

        text = scrape_page(url)

        pages[page_name] = text

        token_count = len(tokenizer.encode(text))

        print(f"\nPagina: {page_name}")
        print(f"URL: {url}")
        print(f"Caratteri: {len(text)}")
        print(f"Token stimati: {token_count}")


    # ===============================
    # SLIDING WINDOW
    # ===============================

    WINDOW_SIZE = 380
    OVERLAP = 80

    page_chunks = {}

    print("\n")
    print("=" * 70)
    print("CREAZIONE SLIDING WINDOW")
    print("=" * 70)


    for page_name, text in pages.items():

        input_ids = tokenizer.encode(text)

        chunks = []

        start = 0

        while start < len(input_ids):

            end = start + WINDOW_SIZE

            chunk_ids = input_ids[start:end]

            chunk_text = tokenizer.decode(
                chunk_ids,
                skip_special_tokens=True
            )

            chunks.append(chunk_text)

            start += WINDOW_SIZE - OVERLAP


        page_chunks[page_name] = chunks


        print(f"\nPagina: {page_name}")
        print(f"Numero chunk: {len(chunks)}")


        for i, chunk in enumerate(chunks):

            print(
                f"Chunk {i+1}: "
                f"{len(tokenizer.encode(chunk))} token"
            )


    return page_chunks

# 3. Estrazione delle informazioni mediante LLM (zero-shot inference)

In questa sezione viene definita la funzione di inferenza utilizzata per estrarre le variabili definite nel codebook mediante prompt schema-guided.

In [60]:
def inferenza_zero_shot(page_chunks, tokenizer, model, device):

    # ==========================================================
    # FRAMEWORK
    # ==========================================================

    framework_attributi = {

        "festival_name":
        """What is the official full name of the festival? Ignore names of associations, organizers,
        partners, sponsors, awards, exhibitions or other events. Answer only with the festival name.""",


        "city_name":
        """Extract only the city where the festival takes place. Ignore cities mentioned as partners,
        sponsors, references, examples, other festivals or external organizations.
        Answer only with the city name.""",


        "first_edition_year":
        """Extract the year when this festival was founded or when its first edition took place.
        Use only information explicitly stated in the context. Do not infer, estimate or guess the year.
        Ignore years related to:
        - organizers or association
        - previous events or related projects
        - artists, exhibitions or awards
        - individual editions after the first edition
        If the context does not explicitly state the foundation year or the first edition year of this
        festival, answer Unknown. Answer only the four-digit year.""",


        "organizer_type":
        """Identify the legal nature of the organization that directly organizes the festival.
        IMPORTANT:
        Classify only the entity explicitly responsible for organizing, curating or managing the festival.
        Do not classify municipalities, public institutions, sponsors, partners, supporters, venues or
        institutions providing patronage.
        If an association, foundation or cultural nonprofit organizes the festival, answer Non-profit
        even if public institutions collaborate or provide support.
        Use only these categories: Public, Private, Non-profit, Mixed. Answer only one category.""",


        "ticket_policy":
        """Identify the admission policy of the festival as a whole. Ignore individual workshops,
        side events, educational activities or special events. If all main festival activities are free,
        answer Free. If all main festival activities require payment, answer Paid. If the festival
        combines free and paid access options, answer Mixed. Answer only one category."""

    }


    # ==========================================================
    # INFERENZA
    # ==========================================================

    from collections import Counter

    dataset = {}

    print("\n")
    print("="*70)
    print("ZERO SHOT INFORMATION EXTRACTION")
    print("="*70)


    for variabile, domanda in framework_attributi.items():

        print("\n")
        print("="*60)
        print(variabile.upper())
        print("="*60)


        risposte_pagine = []


        # --------------------------------------------------
        # Analisi delle pagine
        # --------------------------------------------------

        for page_name, chunks in page_chunks.items():

            print(f"\nPagina: {page_name}")

            risposte_chunk = []


            for i, chunk in enumerate(chunks):

                prompt = f"""

Context:

{chunk}


Question:

{domanda}


Answer:

"""


                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                    truncation=True,
                    max_length=512
                ).to(device)


                with torch.no_grad():

                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=20
                    )


                risposta = tokenizer.decode(
                    outputs[0],
                    skip_special_tokens=True
                ).strip()


                risposte_chunk.append(risposta)


                print(
                    f"Chunk {i+1}: {risposta}"
                )


            # ==================================================
            # SALVATAGGIO RISPOSTA DELLA PAGINA
            # ==================================================

            risposte_pulite = [

                r.strip()

                for r in risposte_chunk

                if r.strip().lower() not in [
                    "",
                    "unknown",
                    "none",
                    "not available",
                    "n/a"
                ]

            ]


            # ==================================================
# RISPOSTA DELLA PAGINA
# ==================================================

            if variabile == "ticket_policy":

                presenti = set(risposte_pulite)

                if "Mixed" in presenti:

                    risposta_pagina = "Mixed"

                elif "Free" in presenti and "Paid" in presenti:

                    risposta_pagina = "Mixed"

                elif "Free" in presenti:

                    risposta_pagina = "Free"

                elif "Paid" in presenti:

                    risposta_pagina = "Paid"

                else:

                    risposta_pagina = "Unknown"

            else:

                if risposte_pulite:

                    risposta_pagina = Counter(
                        risposte_pulite
                    ).most_common(1)[0][0]

                else:

                    risposta_pagina = "Unknown"



            risposte_pagine.append(
                risposta_pagina
            )


            print(
                "Risposta pagina:",
                risposta_pagina
            )



        # ==================================================
        # AGGREGAZIONE TRA PAGINE
        # ==================================================

        print("\nRisposte raccolte dalle pagine:")


        for r in risposte_pagine:

            print("-", r)



        risposte_finali = [

            r

            for r in risposte_pagine

            if r != "Unknown"

        ]



        if not risposte_finali:

            valore_finale = "Unknown"

        elif variabile == "ticket_policy":

            presenti = set(risposte_finali)

            if "Mixed" in presenti:

                valore_finale = "Mixed"

            elif "Free" in presenti and "Paid" in presenti:

                valore_finale = "Mixed"

            elif "Free" in presenti:

                valore_finale = "Free"

            elif "Paid" in presenti:

                valore_finale = "Paid"

            else:

                valore_finale = "Unknown"

        else:

            valore_finale = Counter(
                risposte_finali
            ).most_common(1)[0][0]



        dataset[variabile] = valore_finale



        print(
            "\nRISULTATO FINALE:",
            variabile,
            "=",
            valore_finale
        )



    # ==========================================================
    # OUTPUT DATASET
    # ==========================================================

    df_final = pd.DataFrame(
        [dataset]
    )


    return df_final

# 4. Normalizzazione e arricchimento geografico

La funzione consente di arricchire i dati estratti mediante collegamento con fonti geografiche esterne.

In [117]:
from difflib import get_close_matches


def lookup_istat(df_final, path_istat):

    import pandas as pd


    # ==========================================================
    # CARICAMENTO ISTAT
    # ==========================================================

    istat = pd.read_excel(
        path_istat
    )


    istat.columns = istat.columns.str.strip()


    istat = istat[[

        "Denominazione in italiano",
        "Denominazione Regione",
        "Codice NUTS2 2024 (3)"

    ]]


    # ==========================================================
    # NORMALIZZAZIONE NOMI COMUNI
    # ==========================================================

    istat["nome_norm"] = (

        istat["Denominazione in italiano"]
        .astype(str)
        .str.lower()
        .str.strip()

    )


    city_name = (

        df_final.loc[0, "city_name"]
        .lower()
        .strip()

    )


    # ==========================================================
    # MATCH ESATTO
    # ==========================================================

    match = istat[

        istat["nome_norm"] == city_name

    ]


    # ==========================================================
    # FUZZY MATCH SE FALLISCE
    # ==========================================================

    if match.empty:


        risultato = get_close_matches(

            city_name,
            istat["nome_norm"].tolist(),
            n=1,
            cutoff=0.8

        )


        if risultato:

            nome_trovato = risultato[0]


            print(
                "Fuzzy match:",
                city_name,
                "->",
                nome_trovato
            )


            match = istat[

                istat["nome_norm"] == nome_trovato

            ]



    # ==========================================================
    # OUTPUT
    # ==========================================================

    if not match.empty:


        df_final["region"] = (

            match.iloc[0]["Denominazione Regione"]

        )


        df_final["region_nuts"] = (

            match.iloc[0]["Codice NUTS2 2024 (3)"]

        )

        df_final["city_name"] = (
            match.iloc[0]["Denominazione in italiano"]
        )


    else:


        df_final["region"] = "Unknown"

        df_final["region_nuts"] = "Unknown"



    print("\n")
    print("="*70)
    print("LOOKUP ISTAT")
    print("="*70)

    print(
        df_final.to_string(index=False)
    )


    return df_final

#5. Applicazione della pipeline ai casi studio

## 5.1 Fotografia Calabria Festival

In [118]:
urls = {

    "homepage":
    "https://www.fotografiacalabriafestival.it/",

    "about":
    "https://www.fotografiacalabriafestival.it/pensiero-paesaggio",

    "program":
    "https://www.fotografiacalabriafestival.it/ita/programma"

}


# ==========================================================
# HARVESTING + CHUNKING
# ==========================================================

page_chunks = prepare_pipeline(
    urls
)


# ==========================================================
# ESTRAZIONE DELLE INFORMAZIONI MEDIANTE LLM (ZERO-SHOT)
# ==========================================================

df_result = inferenza_zero_shot(
    page_chunks,
    tokenizer,
    model,
    device
)


# ==========================================================
# ARRICCHIMENTO GEOGRAFICO MEDIANTE LOOKUP ISTAT
# ==========================================================

df_result = lookup_istat(
    df_result,
    ISTAT_PATH
)


# ==========================================================
# VISUALIZZAZIONE DEL RISULTATO ESTRATTO
# ==========================================================

print("RISULTATO PIPELINE: ESTRAZIONE LLM + NORMALIZZAZIONE GEOGRAFICA")

display(df_result)
df_fotografia_calabria = df_result.copy()

HARVESTING DELLE FONTI

Pagina: homepage
URL: https://www.fotografiacalabriafestival.it/
Caratteri: 1166
Token stimati: 442

Pagina: about
URL: https://www.fotografiacalabriafestival.it/pensiero-paesaggio
Caratteri: 5084
Token stimati: 1924

Pagina: program
URL: https://www.fotografiacalabriafestival.it/ita/programma
Caratteri: 4831
Token stimati: 1927


CREAZIONE SLIDING WINDOW

Pagina: homepage
Numero chunk: 2
Chunk 1: 381 token
Chunk 2: 140 token

Pagina: about
Numero chunk: 7
Chunk 1: 381 token
Chunk 2: 381 token
Chunk 3: 381 token
Chunk 4: 381 token
Chunk 5: 381 token
Chunk 6: 381 token
Chunk 7: 125 token

Pagina: program
Numero chunk: 7
Chunk 1: 378 token
Chunk 2: 382 token
Chunk 3: 381 token
Chunk 4: 382 token
Chunk 5: 382 token
Chunk 6: 381 token
Chunk 7: 127 token


ZERO SHOT INFORMATION EXTRACTION


FESTIVAL_NAME

Pagina: homepage
Chunk 1: Fotografia Calabria Festival
Chunk 2: fotografiacalabriafestival.it
Risposta pagina: Fotografia Calabria Festival

Pagina: about
Chunk 1: 

,festival_name,city_name,first_edition_year,organizer_type,ticket_policy,region,region_nuts
0,Fotografia Calabria Festival,San Lucido,2022,Non-profit,Mixed,Calabria,ITF6


## 5.2 Milano Centrale Festival

In [119]:
urls = {

    "homepage":
    " https://www.centralefestival.com/milano2026.html",

    "about":
    "https://www.centralefestival.com/about.html",

    "programme":
    "https://www.centralefestival.com/milano2023.html"

}


# ==========================================================
# HARVESTING + CHUNKING
# ==========================================================

page_chunks = prepare_pipeline(
    urls
)


# ==========================================================
# ESTRAZIONE DELLE INFORMAZIONI MEDIANTE LLM (ZERO-SHOT)
# ==========================================================

df_result = inferenza_zero_shot(
    page_chunks,
    tokenizer,
    model,
    device
)


# ==========================================================
# ARRICCHIMENTO GEOGRAFICO MEDIANTE LOOKUP ISTAT
# ==========================================================

df_result = lookup_istat(
    df_result,
    ISTAT_PATH
)


# ==========================================================
# VISUALIZZAZIONE DEL RISULTATO ESTRATTO
# ==========================================================

print("RISULTATO PIPELINE: ESTRAZIONE LLM + NORMALIZZAZIONE GEOGRAFICA")

display(df_result)
df_milano_centrale = df_result.copy()


HARVESTING DELLE FONTI

Pagina: homepage
URL:  https://www.centralefestival.com/milano2026.html
Caratteri: 2903
Token stimati: 1091

Pagina: about
URL: https://www.centralefestival.com/about.html
Caratteri: 2075
Token stimati: 793

Pagina: programme
URL: https://www.centralefestival.com/milano2023.html
Caratteri: 7236
Token stimati: 2823


CREAZIONE SLIDING WINDOW

Pagina: homepage
Numero chunk: 4
Chunk 1: 379 token
Chunk 2: 381 token
Chunk 3: 382 token
Chunk 4: 191 token

Pagina: about
Numero chunk: 3
Chunk 1: 381 token
Chunk 2: 381 token
Chunk 3: 194 token

Pagina: programme
Numero chunk: 10
Chunk 1: 381 token
Chunk 2: 378 token
Chunk 3: 380 token
Chunk 4: 382 token
Chunk 5: 381 token
Chunk 6: 374 token
Chunk 7: 370 token
Chunk 8: 378 token
Chunk 9: 381 token
Chunk 10: 123 token


ZERO SHOT INFORMATION EXTRACTION


FESTIVAL_NAME

Pagina: homepage
Chunk 1: Milano Centrale Festival
Chunk 2: Milano Centrale Festival
Chunk 3: Milano Centrale Festival
Chunk 4: Associazione Centrale Fotogr

,festival_name,city_name,first_edition_year,organizer_type,ticket_policy,region,region_nuts
0,Milano Centrale Festival,Milano,2023,Non-profit,Free,Lombardia,ITC4


## 5.3 Fotografia Europea

In [120]:
urls = {

    "homepage":
    "https://www.fotografiaeuropea.it/",

    "about":
    "https://www.fotografiaeuropea.it/chi-siamo/",

    "biglietti":
    "https://www.fotografiaeuropea.it/biglietti/"

}


# ==========================================================
# HARVESTING + CHUNKING
# ==========================================================

page_chunks = prepare_pipeline(
    urls
)


# ==========================================================
# ESTRAZIONE DELLE INFORMAZIONI MEDIANTE LLM (ZERO-SHOT)
# ==========================================================

df_result = inferenza_zero_shot(
    page_chunks,
    tokenizer,
    model,
    device
)


# ==========================================================
# ARRICCHIMENTO GEOGRAFICO MEDIANTE LOOKUP ISTAT
# ==========================================================

df_result = lookup_istat(
    df_result,
    ISTAT_PATH
)


# ==========================================================
# VISUALIZZAZIONE DEL RISULTATO ESTRATTO
# ==========================================================

print("RISULTATO PIPELINE: ESTRAZIONE LLM + NORMALIZZAZIONE GEOGRAFICA")

display(df_result)
df_fotografia_europea = df_result.copy()


HARVESTING DELLE FONTI

Pagina: homepage
URL: https://www.fotografiaeuropea.it/
Caratteri: 2866
Token stimati: 1048

Pagina: about
URL: https://www.fotografiaeuropea.it/chi-siamo/
Caratteri: 3976
Token stimati: 1464

Pagina: biglietti
URL: https://www.fotografiaeuropea.it/biglietti/
Caratteri: 4017
Token stimati: 1543


CREAZIONE SLIDING WINDOW

Pagina: homepage
Numero chunk: 4
Chunk 1: 379 token
Chunk 2: 379 token
Chunk 3: 376 token
Chunk 4: 135 token

Pagina: about
Numero chunk: 5
Chunk 1: 381 token
Chunk 2: 382 token
Chunk 3: 378 token
Chunk 4: 380 token
Chunk 5: 250 token

Pagina: biglietti
Numero chunk: 6
Chunk 1: 379 token
Chunk 2: 379 token
Chunk 3: 377 token
Chunk 4: 379 token
Chunk 5: 330 token
Chunk 6: 35 token


ZERO SHOT INFORMATION EXTRACTION


FESTIVAL_NAME

Pagina: homepage
Chunk 1: Fotografia Europea
Chunk 2: 2026 • Fotografia Europea
Chunk 3: una comunicazione su una rete di comunicazione elettronica
Chunk 4: Gestisci consenso Torna in cima
Risposta pagina: Fotografia 

,festival_name,city_name,first_edition_year,organizer_type,ticket_policy,region,region_nuts
0,Fotografia Europea,Reggio nell'Emilia,2027,Non-profit,Free,Emilia-Romagna,ITH5


## 5.4 Venezia Photo

In [121]:
urls = {

    "homepage":
    "https://veneziaphoto.org/it",

    "about":
    "https://veneziaphoto.org/it/pages/infos",

    "programme":
    "https://veneziaphoto.org/it/products/miho-kajioka"


}


# ==========================================================
# HARVESTING + CHUNKING
# ==========================================================

page_chunks = prepare_pipeline(
    urls
)


# ==========================================================
# ESTRAZIONE DELLE INFORMAZIONI MEDIANTE LLM (ZERO-SHOT)
# ==========================================================

df_result = inferenza_zero_shot(
    page_chunks,
    tokenizer,
    model,
    device
)


# ==========================================================
# ARRICCHIMENTO GEOGRAFICO MEDIANTE LOOKUP ISTAT
# ==========================================================

df_result = lookup_istat(
    df_result,
    ISTAT_PATH
)


# ==========================================================
# VISUALIZZAZIONE DEL RISULTATO ESTRATTO
# ==========================================================

print("RISULTATO PIPELINE: ESTRAZIONE LLM + NORMALIZZAZIONE GEOGRAFICA")

display(df_result)
df_venezia_photo = df_result.copy()


HARVESTING DELLE FONTI

Pagina: homepage
URL: https://veneziaphoto.org/it
Caratteri: 4984
Token stimati: 1826

Pagina: about
URL: https://veneziaphoto.org/it/pages/infos
Caratteri: 7411
Token stimati: 2893

Pagina: programme
URL: https://veneziaphoto.org/it/products/miho-kajioka
Caratteri: 10699
Token stimati: 4227


CREAZIONE SLIDING WINDOW

Pagina: homepage
Numero chunk: 7
Chunk 1: 381 token
Chunk 2: 382 token
Chunk 3: 377 token
Chunk 4: 379 token
Chunk 5: 382 token
Chunk 6: 326 token
Chunk 7: 27 token

Pagina: about
Numero chunk: 10
Chunk 1: 381 token
Chunk 2: 381 token
Chunk 3: 381 token
Chunk 4: 381 token
Chunk 5: 381 token
Chunk 6: 381 token
Chunk 7: 382 token
Chunk 8: 381 token
Chunk 9: 380 token
Chunk 10: 193 token

Pagina: programme
Numero chunk: 15
Chunk 1: 381 token
Chunk 2: 381 token
Chunk 3: 382 token
Chunk 4: 380 token
Chunk 5: 379 token
Chunk 6: 376 token
Chunk 7: 372 token
Chunk 8: 380 token
Chunk 9: 380 token
Chunk 10: 382 token
Chunk 11: 380 token
Chunk 12: 379 token


,festival_name,city_name,first_edition_year,organizer_type,ticket_policy,region,region_nuts
0,Venezia Photo,Venezia,2011,Non-profit,Mixed,Veneto,ITH3


# 6. Aggregazione e confronto dei risultati

Gli output prodotti automaticamente dalla pipeline vengono confrontati con i valori presenti nel dataset costruito manualmente, utilizzato come riferimento di validazione.

##6.1 Preparazione dei dataset

In [122]:
# ==========================================================
# CREAZIONE DATASET AUTOMATICO COMPLESSIVO
# ==========================================================

df_automatico = pd.concat(
    [
        df_fotografia_calabria,
        df_milano_centrale,
        df_fotografia_europea,
        df_venezia_photo
    ],
    ignore_index=True
)

display(df_automatico)

,festival_name,city_name,first_edition_year,organizer_type,ticket_policy,region,region_nuts
0,Fotografia Calabria Festival,San Lucido,2022,Non-profit,Mixed,Calabria,ITF6
1,Milano Centrale Festival,Milano,2023,Non-profit,Free,Lombardia,ITC4
2,Fotografia Europea,Reggio nell'Emilia,2027,Non-profit,Free,Emilia-Romagna,ITH5
3,Venezia Photo,Venezia,2011,Non-profit,Mixed,Veneto,ITH3


In [123]:
# ==========================================================
# CARICAMENTO DATASET MANUALE (GOLD STANDARD)
# ==========================================================

df_manuale = pd.read_excel(
    "cultural-data-framework-photography-festivals/dataset/dataset_manual_snapshot_2026.xlsx",
    header=1
)

# pulizia nomi colonne
df_manuale.columns = df_manuale.columns.str.strip()

display(df_manuale.head())

,festival_id,festival_name,city_name,city_geonames_id,region,region_nuts,country_iso,first_edition_year,festival_status,duration_days,...,has_books_press,has_heritage,has_architecture_design,has_performing_arts,has_audiovisual_multimedia,venue_type,ticket_policy,sustainability_claim,notes,record_last_updated
0,FEST000002,ANALOGICA,Bolzano,3181913.0,Trentino-Alto Adige/Südtirol,ITH1,IT,2011,active,5,...,0.0,1.0,0.0,1.0,1.0,cultural_space,unknown,0.0,NaN,2026-07-24
1,FEST000003,Milano Centrale Festival,Milano,6951411.0,Lombardia,ITC4,IT,2023,active,3,...,1.0,0.0,0.0,1.0,1.0,mixed,free,0.0,NaN,2026-07-24
2,FEST000004,Fano Centrale Festival,Fano,3177219.0,Marche,ITI3,IT,2009,active,7,...,1.0,0.0,0.0,0.0,1.0,mixed,free,0.0,NaN,2026-07-24
3,FEST000005,Festival della Fotografia Etica,Lodi,3174638.0,Lombardia,ITC4,IT,2010,active,30,...,1.0,0.0,0.0,0.0,1.0,mixed,paid,1.0,NaN,2026-07-24
4,FEST000006,Siena Awards Photo Festival,Siena,3166548.0,Toscana,ITI1,IT,2015,active,51,...,1.0,0.0,0.0,0.0,0.0,mixed,paid,0.0,NaN,2026-07-24


In [124]:
# ==========================================================
# NORMALIZZAZIONE NOMI FESTIVAL
# ==========================================================

df_manuale["festival_name"] = (
    df_manuale["festival_name"]
    .astype(str)
    .str.strip()
)

df_automatico["festival_name"] = (
    df_automatico["festival_name"]
    .astype(str)
    .str.strip()
)

In [125]:
# ==========================================================
# SELEZIONE CAMPIONE MANUALE
# ==========================================================

festival_test = df_automatico["festival_name"].tolist()

df_manuale_test = df_manuale[
    df_manuale["festival_name"].isin(festival_test)
].copy()

display(df_manuale_test["festival_name"])

,festival_name
1,Milano Centrale Festival
13,Fotografia Calabria Festival
22,Fotografia Europea
77,Venezia Photo


In [126]:
# ==========================================================
# VARIABILI SPERIMENTALI
# ==========================================================

variabili = [
    "festival_name",
    "city_name",
    "first_edition_year",
    "organizer_type",
    "ticket_policy",
    "region",
    "region_nuts"
]

In [127]:
# ==========================================================
# DATASET DI CONFRONTO
# ==========================================================

df_manuale_check = df_manuale_test[variabili].copy()

df_automatico_check = df_automatico[variabili].copy()

In [128]:
df_manuale_check = (
    df_manuale_check
    .sort_values("festival_name")
    .reset_index(drop=True)
)

df_automatico_check = (
    df_automatico_check
    .sort_values("festival_name")
    .reset_index(drop=True)
)

In [129]:
print(df_manuale_check["festival_name"])
print(df_automatico_check["festival_name"])

0    Fotografia Calabria Festival
1              Fotografia Europea
2        Milano Centrale Festival
3                   Venezia Photo
Name: festival_name, dtype: object
0    Fotografia Calabria Festival
1              Fotografia Europea
2        Milano Centrale Festival
3                   Venezia Photo
Name: festival_name, dtype: object


##6.2 Creazione tabella di confronto

La tabella costituisce il supporto per l'analisi qualitativa degli output della pipeline. Le discrepanze individuate sono state successivamente classificate secondo le tipologie di errore discusse nella sezione metodologica.

In [130]:
# ==========================================================
# CREAZIONE TABELLA DI CONFRONTO
# ==========================================================

righe_confronto = []

for i in range(len(df_manuale_check)):

    festival = df_manuale_check.loc[i, "festival_name"]

    for var in variabili:

        valore_manual = df_manuale_check.loc[i, var]
        valore_auto = df_automatico_check.loc[i, var]

        corretto = (
            str(valore_manual).strip().lower()
            ==
            str(valore_auto).strip().lower()
        )

        righe_confronto.append({

            "festival_name": festival,
            "variabile": var,
            "valore_manual": valore_manual,
            "valore_automatico": valore_auto,
            "corretto": corretto

        })


df_confronto = pd.DataFrame(righe_confronto)

display(df_confronto)

,festival_name,variabile,valore_manual,valore_automatico,corretto
0,Fotografia Calabria Festival,festival_name,Fotografia Calabria Festival,Fotografia Calabria Festival,True
1,Fotografia Calabria Festival,city_name,San Lucido,San Lucido,True
2,Fotografia Calabria Festival,first_edition_year,2022,2022,True
3,Fotografia Calabria Festival,organizer_type,non-profit,Non-profit,True
4,Fotografia Calabria Festival,ticket_policy,mixed,Mixed,True
5,Fotografia Calabria Festival,region,Calabria,Calabria,True
6,Fotografia Calabria Festival,region_nuts,ITF6,ITF6,True
7,Fotografia Europea,festival_name,Fotografia Europea,Fotografia Europea,True
8,Fotografia Europea,city_name,Reggio nell'Emilia,Reggio nell'Emilia,True
9,Fotografia Europea,first_edition_year,2006,2027,False
